mount!!

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


Download

In [ ]:
%pip install -q kagglehub

import kagglehub
import os

DATA_DIR = "/content/drive/MyDrive/ML Final/raw"
os.makedirs(DATA_DIR, exist_ok=True)

path = kagglehub.dataset_download(
    "a7madmostafa/us-flight-delays-2025-bts-on-time-performance",
    output_dir=DATA_DIR
)

print("Dataset downloaded to:", path)

100%|██████████| 244M/244M [00:02<00:00, 112MB/s]

Extracting files...


Dataset downloaded to: /content/drive/MyDrive/ML Final/raw


Check CSVs

In [ ]:
from pathlib import Path

csv_files = sorted(Path(path).rglob("*.csv"))

print(f"Found {len(csv_files)} CSV files:")

for file in csv_files:
    print(file.name)

Found 12 CSV files:
flight_delays_2025_01.csv
flight_delays_2025_02.csv
flight_delays_2025_03.csv
flight_delays_2025_04.csv
flight_delays_2025_05.csv
flight_delays_2025_06.csv
flight_delays_2025_07.csv
flight_delays_2025_08.csv
flight_delays_2025_09.csv
flight_delays_2025_10.csv
flight_delays_2025_11.csv
flight_delays_2025_12.csv


Filter

In [ ]:
import pandas as pd

columns = [
    "Year",
    "Month",
    "DayofMonth",
    "DayOfWeek",
    "FlightDate",
    "Reporting_Airline",
    "Origin",
    "Dest",
    "CRSDepTime",
    "CRSArrTime",
    "CRSElapsedTime",
    "Distance",
    "ArrDel15",
    "Cancelled",
    "Diverted"
]

dfs = []

for file in csv_files:
    df = pd.read_csv(file, usecols=columns)
    dfs.append(df)

flights = pd.concat(dfs, ignore_index=True)

print("Total flights:", len(flights))
flights.head()

Total flights: 7001619


,Year,Month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,Origin,Dest,CRSDepTime,CRSArrTime,ArrDel15,Cancelled,Diverted,CRSElapsedTime,Distance
0,2025,1,1,3,2025-01-01,AA,JFK,LAX,659,1020,0.0,0.0,0.0,381.0,2475.0
1,2025,1,2,4,2025-01-02,AA,JFK,LAX,659,1020,0.0,0.0,0.0,381.0,2475.0
2,2025,1,3,5,2025-01-03,AA,JFK,LAX,659,1020,0.0,0.0,0.0,381.0,2475.0
3,2025,1,4,6,2025-01-04,AA,JFK,LAX,700,1026,0.0,0.0,0.0,386.0,2475.0
4,2025,1,5,7,2025-01-05,AA,JFK,LAX,659,1020,0.0,0.0,0.0,381.0,2475.0


In [ ]:
# Ensure filtering columns are numeric
for col in ["Cancelled", "Diverted", "ArrDel15"]:
    flights[col] = pd.to_numeric(flights[col], errors="coerce")

# Keep only completed, non-cancelled, non-diverted flights
flights = flights[
    (flights["Cancelled"] == 0) &
    (flights["Diverted"] == 0) &
    (flights["ArrDel15"].notna())
].copy()

# Target should be an integer (0 or 1)
flights["ArrDel15"] = flights["ArrDel15"].astype(int)

# Remove filtering-only columns
flights.drop(columns=["Cancelled", "Diverted"], inplace=True)

print("Flights remaining:", len(flights))
print("\nTarget distribution:")
print(flights["ArrDel15"].value_counts())
print("\nTarget percentages:")
print(flights["ArrDel15"].value_counts(normalize=True) * 100)

Flights remaining: 6879484

Target distribution:
ArrDel15
0    5344846
1    1534638
Name: count, dtype: int64

Target percentages:
ArrDel15
0    77.692542
1    22.307458
Name: proportion, dtype: float64


Split!!

In [ ]:
%pip install -q pyarrow

PROCESSED_DIR = "/content/drive/MyDrive/ML Final/processed"
os.makedirs(PROCESSED_DIR, exist_ok=True)

processed_path = os.path.join(PROCESSED_DIR, "flights_clean.parquet")

flights.to_parquet(processed_path, index=False)

print("Saved to:", processed_path)

Saved to: /content/drive/MyDrive/ML Final/processed/flights_clean.parquet


Sanity checks

In [ ]:
print("Dataset shape:", flights.shape)
print("\nMissing values:")
print(flights.isnull().sum())

print("\nDate range:")
print(flights["FlightDate"].min(), "to", flights["FlightDate"].max())

print("\nAirlines:", flights["Reporting_Airline"].nunique())
print("Origin airports:", flights["Origin"].nunique())
print("Destination airports:", flights["Dest"].nunique())

print("\nDelayed flight percentage:")
print(f"{flights['ArrDel15'].mean() * 100:.2f}%")

Dataset shape: (6879484, 13)

Missing values:
Year                 0
Month                0
DayofMonth           0
DayOfWeek            0
FlightDate           0
Reporting_Airline    0
Origin               0
Dest                 0
CRSDepTime           0
CRSArrTime           0
ArrDel15             0
CRSElapsedTime       0
Distance             0
dtype: int64

Date range:
2025-01-01 to 2025-12-31

Airlines: 14
Origin airports: 352
Destination airports: 352

Delayed flight percentage:
22.31%


Split test/train

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(flights, groups=flights["FlightDate"])
)

train = flights.iloc[train_idx].copy()
test = flights.iloc[test_idx].copy()

# Save
train.to_parquet(PROCESSED_DIR + "/train.parquet", index=False)
test.to_parquet(PROCESSED_DIR + "/test.parquet", index=False)

print("Train:", len(train))
print("Test:", len(test))

Train: 5533538
Test: 1345946
